# Phase 2.3: Correlation Analysis
**DNA Gene Mapping Project - ML Phase**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Objective
Identify and remove highly correlated (redundant) features

## Key Tasks
1. Compute correlation matrices for numeric features
2. Identify highly correlated pairs (r > 0.95)
3. Select which feature to keep from each pair
4. Check multicollinearity (VIF scores)
5. Create reduced feature set

## Deliverables
- Correlation matrices for each table
- List of redundant features to remove
- VIF analysis
- Correlation heatmaps

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path
import os
from dotenv import load_dotenv
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

PROJECT_ROOT = Path().absolute().parent.parent
REPORTS_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'reports'
FIGURES_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'figures' / 'correlation_analysis'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

print("Setup complete")
print(f"Reports: {REPORTS_DIR}")
print(f"Figures: {FIGURES_DIR}")

In [ ]:
# Database connection
load_dotenv()

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'genome_db')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")

## 1. Load Data for Correlation Analysis

In [ ]:
# Load all 5 feature tables with sampling
print("Loading feature tables for correlation analysis...")

tables = {
    'clinical_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'disease_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'pharmacogene_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'variant_impact_ml_features': 'TABLESAMPLE SYSTEM (5)',
    'structural_variant_ml_features': 'TABLESAMPLE SYSTEM (10)'
}

dataframes = {}

for table_name, sample_clause in tables.items():
    query = f"SELECT * FROM gold.{table_name} {sample_clause}"
    df = pd.read_sql(query, engine)
    dataframes[table_name] = df
    print(f"  {table_name}: {len(df):,} rows, {len(df.columns)} columns")

print(f"\nLoaded {len(dataframes)} tables")

## 2. Compute Correlation Matrices

In [ ]:
# Compute correlation matrices for numeric features only
print("Computing correlation matrices...\n")

correlation_matrices = {}

for table_name, df in dataframes.items():
    # Select numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Exclude ID columns
    numeric_cols = [col for col in numeric_cols if 'id' not in col.lower() and 'pos' not in col.lower()]
    
    if len(numeric_cols) > 1:
        corr_matrix = df[numeric_cols].corr()
        correlation_matrices[table_name] = corr_matrix
        
        print(f"{table_name}:")
        print(f"  Numeric features: {len(numeric_cols)}")
        print(f"  Correlation matrix: {corr_matrix.shape[0]} x {corr_matrix.shape[1]}")
        print()
    else:
        print(f"{table_name}: Insufficient numeric features for correlation\n")

print(f"Computed {len(correlation_matrices)} correlation matrices")

## 3. Identify Highly Correlated Features

In [ ]:
# Find highly correlated feature pairs
CORRELATION_THRESHOLD = 0.95

print(f"Identifying highly correlated pairs (|r| > {CORRELATION_THRESHOLD})...\n")

high_corr_pairs = {}

for table_name, corr_matrix in correlation_matrices.items():
    pairs = []
    
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_value = abs(corr_matrix.iloc[i, j])
            
            if corr_value > CORRELATION_THRESHOLD:
                pairs.append({
                    'Feature 1': corr_matrix.columns[i],
                    'Feature 2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if pairs:
        high_corr_pairs[table_name] = pairs
        print(f"{table_name}:")
        print(f"  High correlation pairs: {len(pairs)}")
        for pair in pairs[:10]:
            print(f"    {pair['Feature 1']:<40} <-> {pair['Feature 2']:<40} (r={pair['Correlation']:.3f})")
        if len(pairs) > 10:
            print(f"    ... and {len(pairs)-10} more pairs")
        print()
    else:
        print(f"{table_name}: No highly correlated pairs found\n")

total_high_corr = sum(len(v) for v in high_corr_pairs.values())
print(f"Total highly correlated pairs: {total_high_corr}")

## 4. Select Features to Remove

In [ ]:
# For each highly correlated pair, select which feature to remove
# Strategy: Keep feature with lower missing values, or alphabetically first if tied

print("Selecting features to remove from correlated pairs...\n")

features_to_remove = {}

for table_name, pairs in high_corr_pairs.items():
    df = dataframes[table_name]
    to_remove = set()
    
    for pair in pairs:
        feat1 = pair['Feature 1']
        feat2 = pair['Feature 2']
        
        # Skip if already marked for removal
        if feat1 in to_remove or feat2 in to_remove:
            continue
        
        # Compare missing value percentages
        missing1 = df[feat1].isnull().sum() / len(df)
        missing2 = df[feat2].isnull().sum() / len(df)
        
        # Remove feature with higher missing percentage
        if missing1 > missing2:
            to_remove.add(feat1)
        elif missing2 > missing1:
            to_remove.add(feat2)
        else:
            # If tied on missing values, keep alphabetically first
            to_remove.add(feat2 if feat1 < feat2 else feat1)
    
    if to_remove:
        features_to_remove[table_name] = list(to_remove)
        print(f"{table_name}:")
        print(f"  Features to remove: {len(to_remove)}")
        for feat in sorted(to_remove):
            print(f"    - {feat}")
        print()

total_to_remove = sum(len(v) for v in features_to_remove.values())
print(f"Total features to remove (correlation): {total_to_remove}")

## 5. Visualize Correlation Matrices

In [ ]:
# Create correlation heatmaps for each table
print("Generating correlation heatmaps...\n")

for table_name, corr_matrix in correlation_matrices.items():
    # Limit to top 30 features by variance for readability
    df = dataframes[table_name]
    numeric_cols = corr_matrix.columns.tolist()
    
    if len(numeric_cols) > 30:
        # Select features with highest variance
        variances = df[numeric_cols].var().sort_values(ascending=False)
        top_features = variances.head(30).index.tolist()
        corr_subset = corr_matrix.loc[top_features, top_features]
        title_suffix = " (Top 30 by variance)"
    else:
        corr_subset = corr_matrix
        title_suffix = ""
    
    fig, ax = plt.subplots(figsize=(14, 12))
    
    sns.heatmap(corr_subset, annot=False, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5,
                cbar_kws={"shrink": 0.8}, ax=ax,
                vmin=-1, vmax=1)
    
    ax.set_title(f'{table_name.replace("_ml_features", "")}{title_suffix}', 
                fontsize=13, fontweight='bold', pad=20)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    
    filename = f"{table_name.replace('_ml_features', '')}_correlation.png"
    plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Saved: {FIGURES_DIR / filename}")

print(f"\nGenerated {len(correlation_matrices)} heatmaps")

## 6. Multicollinearity Analysis (VIF)

In [ ]:
# Compute VIF for clinical_ml_features (most important for variant prediction)
# VIF > 10 indicates high multicollinearity

print("Computing VIF scores for clinical_ml_features...\n")

if 'clinical_ml_features' in dataframes:
    df = dataframes['clinical_ml_features']
    
    # Select numeric features, drop nulls
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if 'id' not in col.lower() and 'pos' not in col.lower()]
    
    # Limit to 20 features to avoid computation time
    if len(numeric_cols) > 20:
        # Select features with highest variance
        variances = df[numeric_cols].var().sort_values(ascending=False)
        numeric_cols = variances.head(20).index.tolist()
    
    df_clean = df[numeric_cols].dropna()
    
    if len(df_clean) > 100:
        vif_data = []
        
        print("Computing VIF (this may take a minute)...")
        
        for i, col in enumerate(df_clean.columns):
            try:
                vif = variance_inflation_factor(df_clean.values, i)
                vif_data.append({'Feature': col, 'VIF': vif})
            except:
                vif_data.append({'Feature': col, 'VIF': np.nan})
        
        vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
        
        print("\nVIF Scores (Top 10):")
        print(vif_df.head(10).to_string(index=False))
        
        high_vif = vif_df[vif_df['VIF'] > 10]
        if len(high_vif) > 0:
            print(f"\nFeatures with high multicollinearity (VIF > 10): {len(high_vif)}")
            print("Recommendation: Consider removing these features")
        else:
            print("\nNo features with high multicollinearity detected")
        
        # Save VIF results
        vif_df.to_csv(REPORTS_DIR / 'vif_analysis.csv', index=False)
        print(f"\nSaved: {REPORTS_DIR / 'vif_analysis.csv'}")
    else:
        print("Insufficient samples for VIF analysis after removing nulls")
else:
    print("clinical_ml_features not available for VIF analysis")

## 7. Summary Statistics

In [ ]:
# Summarize correlation analysis results
summary_data = []

for table_name in dataframes.keys():
    corr_pairs = len(high_corr_pairs.get(table_name, []))
    to_remove = len(features_to_remove.get(table_name, []))
    total_numeric = len(dataframes[table_name].select_dtypes(include=[np.number]).columns)
    
    summary_data.append({
        'Table': table_name.replace('_ml_features', ''),
        'Numeric Features': total_numeric,
        'High Corr Pairs': corr_pairs,
        'To Remove': to_remove,
        'Remaining': total_numeric - to_remove
    })

summary_df = pd.DataFrame(summary_data)

print("\nCorrelation Analysis Summary:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

print(f"\nOverall Statistics:")
print(f"  Total numeric features: {summary_df['Numeric Features'].sum()}")
print(f"  High correlation pairs: {summary_df['High Corr Pairs'].sum()}")
print(f"  Features to remove: {summary_df['To Remove'].sum()}")
print(f"  Features remaining: {summary_df['Remaining'].sum()}")

## 8. Generate Correlation Analysis Report

In [ ]:
# Generate comprehensive correlation analysis report
report_path = REPORTS_DIR / 'correlation_analysis_report.txt'

with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("CORRELATION ANALYSIS REPORT\n")
    f.write("DNA Gene Mapping Project - Phase 2.3\n")
    f.write("="*80 + "\n\n")
    
    f.write("SUMMARY\n")
    f.write("-"*80 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\n")
    
    f.write("OVERALL STATISTICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Total numeric features: {summary_df['Numeric Features'].sum()}\n")
    f.write(f"High correlation pairs (|r| > {CORRELATION_THRESHOLD}): {summary_df['High Corr Pairs'].sum()}\n")
    f.write(f"Features to remove: {summary_df['To Remove'].sum()}\n")
    f.write(f"Features remaining: {summary_df['Remaining'].sum()}\n\n")
    
    f.write("DETAILED FINDINGS\n")
    f.write("="*80 + "\n\n")
    
    if high_corr_pairs:
        f.write(f"HIGHLY CORRELATED FEATURE PAIRS (|r| > {CORRELATION_THRESHOLD})\n")
        f.write("-"*80 + "\n")
        for table, pairs in high_corr_pairs.items():
            f.write(f"\n{table}:\n")
            for pair in pairs:
                f.write(f"  {pair['Feature 1']:<40} <-> {pair['Feature 2']:<40} (r={pair['Correlation']:.3f})\n")
        f.write("\n")
    
    if features_to_remove:
        f.write("FEATURES TO REMOVE (REDUNDANT)\n")
        f.write("-"*80 + "\n")
        for table, features in features_to_remove.items():
            f.write(f"\n{table}:\n")
            for feat in sorted(features):
                f.write(f"  - {feat}\n")
        f.write("\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("RECOMMENDATIONS\n")
    f.write("="*80 + "\n")
    f.write(f"1. Remove {summary_df['To Remove'].sum()} highly correlated features\n")
    f.write("2. Retain features with lower missing values from each pair\n")
    f.write("3. Monitor remaining features for multicollinearity in modeling\n")
    f.write("\n")
    f.write("NEXT STEPS\n")
    f.write("-"*80 + "\n")
    f.write("- Proceed to Phase 2.4: Leakage Detection\n")
    f.write("- Identify features that leak target information\n")
    f.write("- Remove target-derived features\n")

print(f"\nReport saved: {report_path}")
print("\n" + "="*80)
print("PHASE 2.3 COMPLETE - Correlation Analysis")
print("="*80)
print(f"\nGenerated {len(list(FIGURES_DIR.glob('*.png')))} correlation heatmaps")
print(f"Reports: {REPORTS_DIR}")
print("\nCorrelation Findings:")
print(f"  High correlation pairs: {summary_df['High Corr Pairs'].sum()}")
print(f"  Features to remove: {summary_df['To Remove'].sum()}")
print("\nNext: Phase 2.4 - Leakage Detection")